# Programação dinâmica: maior subsequência comum e distância de edição — Tutorial

**Algoritmos e Estruturas de Dados 2 — DCOMP/UFS**
Prof. Dr. André Yoshiaki Kashiwabara

Este tutorial acompanha a aula. O código dos algoritmos é escrito em C e compilado com `gcc`
a partir do próprio notebook: cada célula `%%writefile` grava um programa em `src/`, e a célula
seguinte compila e executa. As medições, os gráficos e alguns exercícios ficam em Python.

## Objetivos

Ao final deste tutorial você será capaz de:

- decidir se uma sequência é **subsequência** de outra em tempo linear e explicar por que a força bruta da LCS é exponencial;
- escrever a recorrência $c[i,j]$ da **LCS** e comparar a recursão direta (exponencial) com a memoizada;
- preencher a tabela $(m+1)\times(n+1)$ **bottom-up** e **reconstruir** uma LCS em $O(m+n)$;
- calcular a **distância de edição** e o alinhamento correspondente, e relacioná-la com a LCS;
- reduzir o espaço para $\Theta(\min(m,n))$ guardando só duas linhas da tabela.

In [ ]:
# Preparação do ambiente: pasta de trabalho, compilador e funções auxiliares.
import os, subprocess

os.makedirs('src', exist_ok=True)
print(subprocess.run(['gcc', '--version'], capture_output=True, text=True).stdout.splitlines()[0])

_compilados = {}   # fonte -> data de modificação da última compilação bem-sucedida

def compilar(fonte):
    '''Compila src/<fonte> com gcc -O2 (só se o arquivo mudou); devolve o executável ou None.'''
    exe = os.path.join('src', os.path.splitext(fonte)[0])
    mtime = os.path.getmtime(os.path.join('src', fonte))
    if _compilados.get(fonte) == mtime and os.path.exists(exe):
        return exe
    comp = subprocess.run(['gcc', '-std=c99', '-Wall', '-O2', '-o', exe,
                           os.path.join('src', fonte), '-lm'],
                          capture_output=True, text=True)
    if comp.returncode != 0:
        print('ERRO DE COMPILAÇÃO:\n' + comp.stderr)
        return None
    if comp.stderr:
        print(comp.stderr)
    _compilados[fonte] = mtime
    return exe

def rodar(fonte, args=(), entrada=None, mostrar=True):
    '''Compila src/<fonte>, executa com os argumentos e a entrada dados e devolve a saída.'''
    exe = compilar(fonte)
    if exe is None:
        return ''
    r = subprocess.run([exe, *map(str, args)], input=entrada, capture_output=True, text=True,
                       errors='replace')
    if mostrar:
        print(r.stdout, end='')
    if r.returncode != 0:
        print(f'(programa terminou com código {r.returncode})', r.stderr)
    return r.stdout

def campo(saida, nome):
    '''Extrai o valor de 'nome=valor' da saída (int se for inteiro, float se for número, senão str).'''
    for tok in saida.split():
        if tok.startswith(nome + '='):
            valor = tok.split('=', 1)[1]
            if valor.lstrip('-').isdigit():
                return int(valor)
            try:
                return float(valor)
            except ValueError:
                return valor
    raise ValueError(f'{nome} não encontrado em: {saida!r}')

def linha(saida, rotulo):
    '''Devolve os inteiros da linha que começa com 'rotulo:' (ex.: 'c3:', 'E5:').'''
    for lin in saida.splitlines():
        if lin.startswith(rotulo + ':'):
            return [int(x) for x in lin.split(':', 1)[1].split()]
    raise ValueError(f'linha {rotulo} não encontrada em: {saida!r}')

# Exemplo condutor da LCS (Cormen et al., 2009, §15.4, Figura 15.8).
X_CLRS, Y_CLRS = 'ABCBDAB', 'BDCABA'
# Exemplo condutor da distância de edição (Dasgupta et al., 2008, §6.3).
X_ED, Y_ED = 'SNOWY', 'SUNNY'

## 1. Subsequências e a força bruta

$Z$ é **subsequência** de $X$ se dá para obter $Z$ apagando símbolos de $X$, sem mudar a ordem
dos que sobram. Uma **LCS** (*longest common subsequence*) de $X$ e $Y$ é uma subsequência comum
de comprimento máximo; pode haver várias. No exemplo do Cormen, $X = \texttt{ABCBDAB}$ e
$Y = \texttt{BDCABA}$ têm LCS de comprimento 4.

A força bruta gera as subsequências de $X$, das mais longas para as mais curtas, e para na
primeira tamanho em que alguma também é subsequência de $Y$. São até $2^m$ candidatas.

**Fonte:** Cormen et al. (2009), §15.4.

In [ ]:
# Exemplo: força bruta da LCS (todas as LCS distintas do exemplo do Cormen).
import random
from itertools import combinations

def eh_subseq_ref(z, y):
    # truque de iterador: cada 'in' consome y até achar o símbolo, então a ordem é respeitada
    it = iter(y)
    return all(s in it for s in z)

def todas_lcs(x, y):
    # testa as subsequências de x do maior comprimento para o menor
    for k in range(len(x), -1, -1):
        achadas = {''.join(c) for c in combinations(x, k) if eh_subseq_ref(c, y)}
        if achadas:
            return k, sorted(achadas)

k, lcs_clrs = todas_lcs(X_CLRS, Y_CLRS)
print(f'comprimento da LCS = {k}; LCS distintas: {lcs_clrs}')
print('SNOWY x SUNNY:', todas_lcs(X_ED, Y_ED))

In [ ]:
# Visualização: tempo da força bruta quando X e Y não têm nada em comum (pior caso: 2^m candidatas).
import time
import matplotlib.pyplot as plt

ms, tempos_fb = list(range(4, 19, 2)), []
for m in ms:
    t0 = time.perf_counter()
    todas_lcs('a' * m, 'b' * m)
    tempos_fb.append(time.perf_counter() - t0)

plt.figure(figsize=(6, 3.5))
plt.semilogy(ms, tempos_fb, 'o-')
plt.xlabel('m = n')
plt.ylabel('tempo (s, escala log)')
plt.title('Força bruta da LCS: cada símbolo a mais dobra o tempo')
plt.grid(True, which='both', alpha=0.3)
plt.show()

In [ ]:
# Exercício 1: subsequência em tempo linear
# TODO: implemente eh_subsequencia(z, y), que devolve True se z é subsequência de y.
# Use dois índices: percorra y uma única vez, avançando em z sempre que o símbolo casar.
# Não use o truque do iterador nem combinations: a ideia é escrever o laço O(len(y)).

def eh_subsequencia(z, y):
    # TODO: implemente aqui
    pass


# Teste automático
assert eh_subsequencia('BCBA', X_CLRS) is not None, 'Implemente eh_subsequencia()'
assert eh_subsequencia('BCBA', X_CLRS) and eh_subsequencia('BCBA', Y_CLRS), 'BCBA é LCS do exemplo do Cormen'
assert not eh_subsequencia('ABBA', Y_CLRS), 'Y = BDCABA tem só um A antes do último B'
assert eh_subsequencia('', 'XYZ') and not eh_subsequencia('A', ''), 'Casos com cadeia vazia'
assert not eh_subsequencia('BA', 'AB'), 'A ordem importa'
random.seed(1)
for _ in range(2000):
    y = ''.join(random.choice('AB') for _ in range(random.randint(0, 8)))
    z = ''.join(random.choice('AB') for _ in range(random.randint(0, 5)))
    assert eh_subsequencia(z, y) == eh_subseq_ref(z, y), f'Falhou com z={z!r}, y={y!r}'
print('Exercício 1 ok!')

## 2. A recorrência: recursão direta e memoization

O subproblema é o par de **prefixos** $X_i = x_1\cdots x_i$ e $Y_j = y_1\cdots y_j$. Pelo
Teorema 15.1 do Cormen, ou os últimos símbolos casam, ou um deles fica fora da LCS:

$$c[i,j] = \begin{cases} 0 & \text{se } i = 0 \text{ ou } j = 0,\\
c[i-1,j-1] + 1 & \text{se } x_i = y_j,\\
\max(c[i-1,j],\ c[i,j-1]) & \text{se } x_i \ne y_j.\end{cases}$$

Sem tabela, a recursão repete subproblemas. O pior caso é quando **nenhum** símbolo casa: toda
chamada se divide em duas. O programa abaixo conta as chamadas da recursão direta e da memoizada
com $X = \texttt{aa}\cdots\texttt{a}$ e $Y = \texttt{bb}\cdots\texttt{b}$.

**Fonte:** Cormen et al. (2009), §15.4 (equação 15.9) e §15.3 (memoization).

In [ ]:
%%writefile src/lcs_rec.c
/* LCS: recursao direta x memoization (contagem de chamadas).
   Uso: ./lcs_rec X Y
   Cormen et al. (2009), sec. 15.4 (equacao 15.9) e sec. 15.3. */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

static const char *x, *y;
static int **T;                    /* T[i][j] == -1: ainda nao calculado */
static long long chamIngenuo = 0, chamMemo = 0;

int lcsIngenuo( int i, int j) {
   ++chamIngenuo;
   if (i == 0 || j == 0) return 0;
   if (x[i-1] == y[j-1]) return lcsIngenuo( i-1, j-1) + 1;
   int a = lcsIngenuo( i-1, j), b = lcsIngenuo( i, j-1);
   return a >= b ? a : b;
}

int lcsMemo( int i, int j) {
   ++chamMemo;
   if (i == 0 || j == 0) return 0;
   if (T[i][j] >= 0) return T[i][j];
   int r;
   if (x[i-1] == y[j-1]) r = lcsMemo( i-1, j-1) + 1;
   else {
      int a = lcsMemo( i-1, j), b = lcsMemo( i, j-1);
      r = a >= b ? a : b;
   }
   return T[i][j] = r;
}

int main( int argc, char *argv[]) {
   if (argc < 3) { fprintf( stderr, "uso: %s X Y\n", argv[0]); return 1; }
   x = argv[1];  y = argv[2];
   int m = strlen( x), n = strlen( y);
   T = malloc( (m+1) * sizeof (int *));
   for (int i = 0; i <= m; ++i) {
      T[i] = malloc( (n+1) * sizeof (int));
      for (int j = 0; j <= n; ++j) T[i][j] = -1;
   }
   int a = lcsIngenuo( m, n), b = lcsMemo( m, n);
   printf( "m=%d n=%d ingenuo=%d memo=%d chamadas_ingenuo=%lld chamadas_memo=%lld\n",
           m, n, a, b, chamIngenuo, chamMemo);
   return 0;
}

In [ ]:
# Exemplo: o exemplo do Cormen e o pior caso (nenhum símbolo em comum) com m = n crescendo.
s = rodar('lcs_rec.c', args=(X_CLRS, Y_CLRS))
assert campo(s, 'ingenuo') == campo(s, 'memo') == 4

cham_ing, cham_memo = {}, {}
for m in range(2, 14):
    s = rodar('lcs_rec.c', args=('a' * m, 'b' * m), mostrar=(m % 4 == 1))
    assert campo(s, 'ingenuo') == campo(s, 'memo') == 0
    assert campo(s, 'chamadas_memo') <= 2 * m * m + 1
    cham_ing[m], cham_memo[m] = campo(s, 'chamadas_ingenuo'), campo(s, 'chamadas_memo')
print('As duas versões concordam em todas as instâncias.')

In [ ]:
# Visualização: chamadas (escala log). A recursão direta faz 2*C(2m, m) - 1 chamadas no pior caso.
from math import comb

ms = sorted(cham_ing)
assert all(cham_ing[m] == 2 * comb(2 * m, m) - 1 for m in ms)
plt.figure(figsize=(6, 4))
plt.semilogy(ms, [cham_ing[m] for m in ms], 'o-', label='recursão direta')
plt.semilogy(ms, [cham_memo[m] for m in ms], 's-', label='memoizada')
plt.semilogy(ms, [4 ** m for m in ms], '--', label=r'$4^m$')
plt.xlabel('m = n (nenhum símbolo em comum)')
plt.ylabel('chamadas (escala log)')
plt.title('LCS: recursão direta x memoization')
plt.legend()
plt.grid(True, which='both', alpha=0.3)
plt.show()

### Exercício 2: a tabela bottom-up

Complete `lcsComprimento()`: ela deve preencher `c[0..m][0..n]` linha por linha, como nos slides,
e devolver `c[m][n]`. Lembre que em C a cadeia começa no índice 0: $x_i$ é `x[i-1]`. O `main`
compara a sua resposta com a força bruta em 500 pares aleatórios sobre o alfabeto `{A, C, G, T}`.

In [ ]:
%%writefile src/lcs_ex2.c
/* Exercicio 2: LCS BOTTOM-UP.
   c[i][j] = comprimento de uma LCS de x[0..i-1] e y[0..j-1]:
      c[i][0] = c[0][j] = 0,
      c[i][j] = c[i-1][j-1] + 1                  se x[i-1] == y[j-1],
      c[i][j] = max( c[i-1][j], c[i][j-1])       caso contrario.
   TODO: complete lcsComprimento(). O main confere com a forca bruta. */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

#define MAX 14

static int c[MAX+1][MAX+1];

int lcsComprimento( const char x[], int m, const char y[], int n) {
   /* TODO: preencha c[0..m][0..n] e devolva c[m][n] */
   return -1;
}

/* z (comprimento k) e subsequencia de y (comprimento n)? */
static int ehSubseq( const char z[], int k, const char y[], int n) {
   int p = 0;
   for (int j = 0; j < n && p < k; ++j) if (z[p] == y[j]) ++p;
   return p == k;
}

/* testa os 2^m subconjuntos de posicoes de x */
static int forcaBruta( const char x[], int m, const char y[], int n) {
   char z[MAX+1];
   int melhor = 0;
   for (int S = 0; S < (1 << m); ++S) {
      int k = 0;
      for (int i = 0; i < m; ++i) if (S & (1 << i)) z[k++] = x[i];
      if (k > melhor && ehSubseq( z, k, y, n)) melhor = k;
   }
   return melhor;
}

int main( void) {
   char x[MAX+1], y[MAX+1];
   printf( "clrs=%d\n", lcsComprimento( "ABCBDAB", 7, "BDCABA", 6));
   srand( 2026);
   for (int teste = 0; teste < 500; ++teste) {
      int m = rand() % (MAX + 1), n = rand() % (MAX + 1);
      for (int i = 0; i < m; ++i) x[i] = "ACGT"[rand() % 4];
      for (int j = 0; j < n; ++j) y[j] = "ACGT"[rand() % 4];
      x[m] = y[n] = '\0';
      int a = lcsComprimento( x, m, y, n), b = forcaBruta( x, m, y, n);
      if (a != b) {
         printf( "ERRO: x=%s y=%s, esperado %d, obtido %d\nok=0\n", x, y, b, a);
         return 0;
      }
   }
   printf( "ok=1\n");
   return 0;
}

In [ ]:
# Teste automático do Exercício 2
saida = rodar('lcs_ex2.c')
assert campo(saida, 'clrs') == 4, 'No exemplo do Cormen, c[7][6] = 4'
assert campo(saida, 'ok') == 1, 'Verifique sua implementação (veja o ERRO impresso acima)'
print('Exercício 2 ok!')

## 3. A tabela completa e a reconstrução

Com a tabela pronta, uma LCS sai **de trás para frente**, a partir de $(m,n)$: se $x_i = y_j$, o
símbolo entra na LCS e andamos na diagonal; senão, vamos para a vizinha (acima ou à esquerda) de
onde veio o máximo. Em caso de empate, o Cormen sobe; por isso o exemplo dá `BCBA`, e não outra
das LCS encontradas pela força bruta.

O programa lê duas cadeias, imprime a tabela $c$ linha a linha e a LCS reconstruída.

**Fonte:** Cormen et al. (2009), §15.4, LCS-LENGTH, PRINT-LCS e Figura 15.8.

In [ ]:
%%writefile src/lcs.c
/* LCS por programacao dinamica: tabela bottom-up e reconstrucao.
   Uso: ./lcs < entrada     (duas linhas: X e Y; imprime a tabela, o comprimento e uma LCS)
   Cormen et al. (2009), sec. 15.4. */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

#define MAX 5000

static char x[MAX+2], y[MAX+2];
static int **c;

int lcs( int m, int n) {
   for (int i = 0; i <= m; ++i) c[i][0] = 0;
   for (int j = 0; j <= n; ++j) c[0][j] = 0;
   for (int i = 1; i <= m; ++i)
      for (int j = 1; j <= n; ++j)
         if (x[i-1] == y[j-1])
            c[i][j] = c[i-1][j-1] + 1;      /* casa x_i com y_j */
         else if (c[i-1][j] >= c[i][j-1])
            c[i][j] = c[i-1][j];            /* descarta x_i     */
         else
            c[i][j] = c[i][j-1];            /* descarta y_j     */
   return c[m][n];
}

/* imprime uma LCS de x[0..i-1] e y[0..j-1] (PRINT-LCS do Cormen) */
void imprime( int i, int j) {
   if (i == 0 || j == 0) return;
   if (x[i-1] == y[j-1]) {
      imprime( i-1, j-1);
      putchar( x[i-1]);
   } else if (c[i-1][j] >= c[i][j-1])
      imprime( i-1, j);
   else
      imprime( i, j-1);
}

static void leLinha( char s[]) {
   if (fgets( s, MAX+2, stdin) == NULL) s[0] = '\0';
   s[strcspn( s, "\r\n")] = '\0';
}

int main( void) {
   leLinha( x);  leLinha( y);
   int m = strlen( x), n = strlen( y);
   c = malloc( (m+1) * sizeof (int *));
   for (int i = 0; i <= m; ++i) c[i] = malloc( (n+1) * sizeof (int));
   int k = lcs( m, n);
   if (m <= 20 && n <= 20)
      for (int i = 0; i <= m; ++i) {
         printf( "c%d:", i);
         for (int j = 0; j <= n; ++j) printf( " %d", c[i][j]);
         printf( "\n");
      }
   printf( "comprimento=%d\nlcs=", k);
   imprime( m, n);
   printf( "\n");
   return 0;
}

In [ ]:
# Exemplo: o exemplo condutor dos slides (Cormen, Figura 15.8).
saida = rodar('lcs.c', entrada=f'{X_CLRS}\n{Y_CLRS}\n')
assert linha(saida, 'c7') == [0, 1, 2, 2, 3, 4, 4]
assert campo(saida, 'comprimento') == 4 and campo(saida, 'lcs') == 'BCBA'
assert campo(saida, 'lcs') in lcs_clrs
print('Exemplo do Cormen conferido: c[7][6] = 4, LCS = BCBA.')

In [ ]:
# Visualização: a tabela c como mapa de calor, com o caminho da reconstrução.
import numpy as np

c = np.array([linha(saida, f'c{i}') for i in range(len(X_CLRS) + 1)])
caminho, i, j = [], len(X_CLRS), len(Y_CLRS)
while i > 0 and j > 0:
    caminho.append((i, j))
    if X_CLRS[i - 1] == Y_CLRS[j - 1]:
        i, j = i - 1, j - 1
    elif c[i - 1][j] >= c[i][j - 1]:
        i -= 1
    else:
        j -= 1
caminho.append((i, j))

plt.figure(figsize=(5.5, 5))
plt.imshow(c, cmap='Blues')
for a in range(c.shape[0]):
    for b in range(c.shape[1]):
        plt.text(b, a, c[a, b], ha='center', va='center',
                 color='white' if c[a, b] >= 3 else 'black')
ys, xs = zip(*caminho)
plt.plot(xs, ys, 'o-', color='tab:orange', lw=2, ms=20, mfc='none', mew=2)
plt.xticks(range(len(Y_CLRS) + 1), [''] + list(Y_CLRS))
plt.yticks(range(len(X_CLRS) + 1), [''] + list(X_CLRS))
plt.title('Tabela c do Cormen (em laranja: a reconstrução)')
plt.show()

### Exercício 3: reconstrução iterativa

A `imprime()` dos slides é recursiva e usa $O(m+n)$ de pilha. Complete `reconstroi()`, que faz o
mesmo **sem recursão**: grava em `z[]` uma LCS (terminada em `'\0'`) e devolve o seu comprimento.
Dica: percorra de $(m,n)$ até a borda gravando os símbolos **de trás para frente** em
`z[c[m][n]-1]`, `z[c[m][n]-2]`, … Use o mesmo desempate do Cormen (sobe quando
$c[i-1][j] \ge c[i][j-1]$) para obter `BCBA` no exemplo. O `main` confere, em 500 pares
aleatórios, que `z` é subsequência das duas cadeias e tem o comprimento certo.

In [ ]:
%%writefile src/lcs_ex3.c
/* Exercicio 3: RECONSTRUCAO ITERATIVA de uma LCS a partir da tabela c.
   tabela() preenche c[0..m][0..n] como nos slides (ja esta pronta).
   TODO: complete reconstroi(): grave em z[] uma LCS de x e y, terminada em '\0', e devolva
   seu comprimento. Sem recursao: O(m+n) passos e O(1) de memoria extra. */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

#define MAX 300

static int c[MAX+1][MAX+1];

void tabela( const char x[], int m, const char y[], int n) {
   for (int i = 0; i <= m; ++i) c[i][0] = 0;
   for (int j = 0; j <= n; ++j) c[0][j] = 0;
   for (int i = 1; i <= m; ++i)
      for (int j = 1; j <= n; ++j)
         if (x[i-1] == y[j-1]) c[i][j] = c[i-1][j-1] + 1;
         else c[i][j] = c[i-1][j] >= c[i][j-1] ? c[i-1][j] : c[i][j-1];
}

int reconstroi( const char x[], int m, const char y[], int n, char z[]) {
   int k = c[m][n];
   z[k] = '\0';
   /* TODO: ande de (m,n) ate a borda, gravando os simbolos casados em z[k-1], z[k-2], ... */
   return k;
}

static int ehSubseq( const char z[], const char y[]) {
   int p = 0;
   for (int j = 0; y[j] != '\0' && z[p] != '\0'; ++j) if (z[p] == y[j]) ++p;
   return z[p] == '\0';
}

int main( void) {
   char x[MAX+1], y[MAX+1], z[MAX+1];
   tabela( "ABCBDAB", 7, "BDCABA", 6);
   memset( z, '?', sizeof z);  z[MAX] = '\0';
   reconstroi( "ABCBDAB", 7, "BDCABA", 6, z);
   printf( "clrs=%s\n", z);
   srand( 2026);
   for (int teste = 0; teste < 500; ++teste) {
      int m = rand() % (MAX + 1), n = rand() % (MAX + 1);
      for (int i = 0; i < m; ++i) x[i] = "ACGT"[rand() % 4];
      for (int j = 0; j < n; ++j) y[j] = "ACGT"[rand() % 4];
      x[m] = y[n] = '\0';
      tabela( x, m, y, n);
      memset( z, '?', sizeof z);
      int k = reconstroi( x, m, y, n, z);
      z[MAX] = '\0';
      if (k != c[m][n] || (int) strlen( z) != k || !ehSubseq( z, x) || !ehSubseq( z, y)) {
         printf( "ERRO: m=%d n=%d, esperado comprimento %d; obtido k=%d"
                 " (z e subseq de x? %d; de y? %d)\nok=0\n", m, n, c[m][n], k,
                 ehSubseq( z, x), ehSubseq( z, y));
         return 0;
      }
   }
   printf( "ok=1\n");
   return 0;
}

In [ ]:
# Teste automático do Exercício 3
saida = rodar('lcs_ex3.c')
assert campo(saida, 'clrs') == 'BCBA', 'Com o desempate do Cormen, a LCS do exemplo é BCBA'
assert campo(saida, 'ok') == 1, 'Verifique sua implementação (veja o ERRO impresso acima)'
print('Exercício 3 ok!')

## 4. Distância de edição

A **distância de edição** (Levenshtein) é o menor número de **inserções**, **remoções** e
**substituições** de um símbolo que transformam $X$ em $Y$. Pensando no alinhamento de $X$ sobre
$Y$, a última coluna é uma de três, e isso dá a recorrência de Wagner e Fischer:

$$E[i,j] = \min\bigl(E[i-1,j] + 1,\ \ E[i,j-1] + 1,\ \ E[i-1,j-1] + \mathrm{dif}(x_i,y_j)\bigr),
\qquad E[i,0] = i,\ \ E[0,j] = j.$$

O programa abaixo imprime a tabela $E$ e reconstrói **um** alinhamento ótimo. Na volta, ele
prefere remover, depois inserir, depois a diagonal; com esse desempate, sai o alinhamento dos
slides, `S-NOWY` sobre `SUNN-Y`. Outros desempates dão outros alinhamentos, também de custo 3.

**Fonte:** Dasgupta et al. (2008), §6.3; Wagner e Fischer (1974); Cormen et al. (2009), Problema 15-5.

In [ ]:
%%writefile src/edicao.c
/* DISTANCIA DE EDICAO (Levenshtein) por programacao dinamica, com o alinhamento.
   Uso: ./edicao < entrada     (duas linhas: X e Y)
   Dasgupta et al. (2008), sec. 6.3; Wagner e Fischer (1974). */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

#define MAX 5000

static char x[MAX+2], y[MAX+2];
static int **E;

int edicao( int m, int n) {
   for (int i = 0; i <= m; ++i) E[i][0] = i;   /* remove tudo */
   for (int j = 0; j <= n; ++j) E[0][j] = j;   /* insere tudo */
   for (int i = 1; i <= m; ++i)
      for (int j = 1; j <= n; ++j) {
         int r = E[i-1][j] + 1;                         /* remove x_i */
         if (E[i][j-1] + 1 < r) r = E[i][j-1] + 1;      /* insere y_j */
         int d = E[i-1][j-1] + (x[i-1] != y[j-1]);      /* casa/troca */
         E[i][j] = d < r ? d : r;
      }
   return E[m][n];
}

/* refaz as decisoes de (m,n) ate (0,0); monta as duas linhas do alinhamento ao contrario */
void alinhamento( int m, int n) {
   char *a = malloc( m + n + 1), *b = malloc( m + n + 1);
   int k = 0, i = m, j = n;
   while (i > 0 || j > 0) {
      if (i > 0 && E[i][j] == E[i-1][j] + 1) { a[k] = x[--i]; b[k++] = '-'; }
      else if (j > 0 && E[i][j] == E[i][j-1] + 1) { a[k] = '-'; b[k++] = y[--j]; }
      else { a[k] = x[--i]; b[k++] = y[--j]; }
   }
   printf( "alinX=");
   for (int t = k-1; t >= 0; --t) putchar( a[t]);
   printf( "\nalinY=");
   for (int t = k-1; t >= 0; --t) putchar( b[t]);
   printf( "\n");
   free( a);  free( b);
}

static void leLinha( char s[]) {
   if (fgets( s, MAX+2, stdin) == NULL) s[0] = '\0';
   s[strcspn( s, "\r\n")] = '\0';
}

int main( void) {
   leLinha( x);  leLinha( y);
   int m = strlen( x), n = strlen( y);
   E = malloc( (m+1) * sizeof (int *));
   for (int i = 0; i <= m; ++i) E[i] = malloc( (n+1) * sizeof (int));
   int d = edicao( m, n);
   if (m <= 20 && n <= 20)
      for (int i = 0; i <= m; ++i) {
         printf( "E%d:", i);
         for (int j = 0; j <= n; ++j) printf( " %d", E[i][j]);
         printf( "\n");
      }
   printf( "distancia=%d\n", d);
   alinhamento( m, n);
   return 0;
}

In [ ]:
# Exemplo: SNOWY -> SUNNY (Dasgupta, §6.3) e algoritimo -> algoritmo.
saida = rodar('edicao.c', entrada=f'{X_ED}\n{Y_ED}\n')
assert linha(saida, 'E5') == [5, 4, 4, 4, 4, 3]
assert campo(saida, 'distancia') == 3
assert (campo(saida, 'alinX'), campo(saida, 'alinY')) == ('S-NOWY', 'SUNN-Y')
print()
s = rodar('edicao.c', entrada='algoritimo\nalgoritmo\n', mostrar=False)
assert campo(s, 'distancia') == 1
print('d(algoritimo, algoritmo) =', campo(s, 'distancia'), '|', campo(s, 'alinX'), 'sobre', campo(s, 'alinY'))

In [ ]:
# Visualização: distância de edição x distância só com inserções e remoções (m + n - 2|LCS|),
# em 300 pares aleatórios de DNA de comprimento 30.
def lcs_py(x, y):
    ant = [0] * (len(y) + 1)
    for a in x:
        atu = [0]
        for j, b in enumerate(y, 1):
            atu.append(ant[j - 1] + 1 if a == b else max(ant[j], atu[j - 1]))
        ant = atu
    return ant[-1]

def edicao_py(x, y):
    ant = list(range(len(y) + 1))
    for i, a in enumerate(x, 1):
        atu = [i]
        for j, b in enumerate(y, 1):
            atu.append(min(ant[j] + 1, atu[j - 1] + 1, ant[j - 1] + (a != b)))
        ant = atu
    return ant[-1]

random.seed(2026)
pares = [(''.join(random.choices('ACGT', k=30)), ''.join(random.choices('ACGT', k=30)))
         for _ in range(300)]
d_lev = [edicao_py(x, y) for x, y in pares]
d_ir = [len(x) + len(y) - 2 * lcs_py(x, y) for x, y in pares]
assert all(d <= e <= 2 * d for d, e in zip(d_lev, d_ir))   # trocar custa no máximo 1 remoção + 1 inserção

plt.figure(figsize=(5.5, 4))
plt.scatter(d_lev, d_ir, alpha=0.4)
lim = [min(d_lev), max(d_lev)]
plt.plot(lim, lim, '--', label='$y = x$')
plt.plot(lim, [2 * v for v in lim], ':', label='$y = 2x$')
plt.xlabel('distância de edição (com substituição)')
plt.ylabel('$m + n - 2|LCS|$ (só inserção/remoção)')
plt.title('As duas distâncias em pares aleatórios de DNA')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Exercício 4: distância de edição com duas linhas

Para obter só a **distância**, basta guardar a linha anterior e a atual da tabela: espaço
$\Theta(n)$ em vez de $\Theta(mn)$. Complete `edicao2L()`. Cuidado com a primeira coluna de cada
linha: $E[i,0] = i$. O `main` compara com a versão de tabela inteira em 500 pares aleatórios e
depois calcula a distância entre duas cadeias de 20000 símbolos, que pela tabela inteira
precisariam de $\approx 1{,}6$ GB.

In [ ]:
%%writefile src/edicao_ex4.c
/* Exercicio 4: DISTANCIA DE EDICAO COM DUAS LINHAS (espaco O(n)).
   ant[j] guarda E[i-1][j]; atu[j] deve receber E[i][j].
   TODO: complete edicao2L(). O main compara com a tabela inteira em instancias aleatorias. */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

int edicao2L( const char x[], int m, const char y[], int n) {
   int *ant = malloc( (n+1) * sizeof (int)), *atu = malloc( (n+1) * sizeof (int));
   for (int j = 0; j <= n; ++j) ant[j] = j;              /* linha i = 0 */
   /* TODO: para i = 1..m, preencha atu[0..n] a partir de ant[] e troque os papeis */
   int r = ant[n];
   free( ant);  free( atu);
   return r;
}

#define MAX 60
static int E[MAX+1][MAX+1];

static int tabelaInteira( const char x[], int m, const char y[], int n) {
   for (int i = 0; i <= m; ++i) E[i][0] = i;
   for (int j = 0; j <= n; ++j) E[0][j] = j;
   for (int i = 1; i <= m; ++i)
      for (int j = 1; j <= n; ++j) {
         int r = E[i-1][j] + 1;
         if (E[i][j-1] + 1 < r) r = E[i][j-1] + 1;
         int d = E[i-1][j-1] + (x[i-1] != y[j-1]);
         E[i][j] = d < r ? d : r;
      }
   return E[m][n];
}

int main( void) {
   char x[MAX+1], y[MAX+1];
   printf( "snowy=%d\n", edicao2L( "SNOWY", 5, "SUNNY", 5));
   srand( 2026);
   for (int teste = 0; teste < 500; ++teste) {
      int m = rand() % (MAX + 1), n = rand() % (MAX + 1);
      for (int i = 0; i < m; ++i) x[i] = "ACGT"[rand() % 4];
      for (int j = 0; j < n; ++j) y[j] = "ACGT"[rand() % 4];
      x[m] = y[n] = '\0';
      int a = edicao2L( x, m, y, n), b = tabelaInteira( x, m, y, n);
      if (a != b) {
         printf( "ERRO: x=%s y=%s, esperado %d, obtido %d\nok=0\n", x, y, b, a);
         return 0;
      }
   }
   /* cadeias grandes: y e x com ~10% dos simbolos trocados */
   int N = 20000;
   char *X = malloc( N + 1), *Y = malloc( N + 1);
   for (int i = 0; i < N; ++i) {
      X[i] = "ACGT"[rand() % 4];
      Y[i] = rand() % 10 == 0 ? "ACGT"[rand() % 4] : X[i];
   }
   X[N] = Y[N] = '\0';
   clock_t t0 = clock();
   int d = edicao2L( X, N, Y, N);
   printf( "grande=%d tempo=%.2f\nok=1\n", d, (double) (clock() - t0) / CLOCKS_PER_SEC);
   return 0;
}

In [ ]:
# Teste automático do Exercício 4
saida = rodar('edicao_ex4.c')
assert campo(saida, 'snowy') == 3, 'd(SNOWY, SUNNY) = 3'
assert campo(saida, 'ok') == 1, 'Verifique sua implementação (veja o ERRO impresso acima)'
assert 0 < campo(saida, 'grande') <= 20000
print('Exercício 4 ok!')

## Desafio Final: alinhamento de DNA com custos

Em sequências de DNA, as **transições** (troca entre purinas, `A`↔`G`, ou entre pirimidinas,
`C`↔`T`) são mais frequentes que as **transversões** (purina ↔ pirimidina). O alinhamento de
Kleinberg e Tardos (§6.6) usa um custo $\delta$ por lacuna e um custo $\alpha_{pq}$ por alinhar
$p$ com $q$:

$$\mathrm{OPT}(i,j) = \min\bigl(\alpha_{x_i y_j} + \mathrm{OPT}(i-1,j-1),\ \ \delta + \mathrm{OPT}(i-1,j),\ \ \delta + \mathrm{OPT}(i,j-1)\bigr).$$

Use $\delta = 2$, $\alpha_{pp} = 0$, $\alpha = 1$ para transições e $\alpha = 3$ para transversões.

1. **Casos base e ordem.** Quanto valem $\mathrm{OPT}(i,0)$ e $\mathrm{OPT}(0,j)$? Em que ordem a
   tabela pode ser preenchida?
2. **Implementação.** Escreva `src/alinhamento.c`, que lê duas sequências e imprime o custo mínimo
   e o alinhamento (duas linhas com `-` nas lacunas), em $\Theta(mn)$.
3. **Verificação.** Com $\delta = 1$ e $\alpha_{pq} = 1$ para $p \ne q$, o seu programa deve dar a
   distância de edição: confira com `src/edicao.c` em `SNOWY`/`SUNNY` e em pares aleatórios.
4. **Experimento.** Alinhe `GATTACAGATTACA` com `GACTATAGGTTCA`. O alinhamento ótimo com os custos
   acima é o mesmo que o da distância de edição? Explique a diferença olhando para as colunas.
5. **Para ir além.** Com custos que podem ser **negativos** para casamentos (pontuação de
   similaridade) e maximizando em vez de minimizar, obtém-se o algoritmo de Needleman e Wunsch
   (1970). O que muda na recorrência?

In [ ]:
# Desafio: alinhamento de DNA com custos
# TODO: resolva o desafio abaixo
# Sugestão: comece copiando src/edicao.c para src/alinhamento.c e troque os custos fixos por
# uma função alfa(p, q) e uma constante DELTA.

# Sua solução aqui

## Referências

Veja o arquivo `../referencias.bib` para a lista completa. As principais para este tutorial:

- CORMEN, T. H. et al. *Introduction to Algorithms*. 3. ed. MIT Press, 2009. §15.4 e Problema 15-5. (Em português: *Algoritmos: Teoria e Prática*, Elsevier, 2012.)
- DASGUPTA, S.; PAPADIMITRIOU, C. H.; VAZIRANI, U. *Algorithms*. McGraw-Hill, 2008. §6.3.
- KLEINBERG, J.; TARDOS, É. *Algorithm Design*. Pearson/Addison-Wesley, 2006. §§6.6–6.7.
- WAGNER, R. A.; FISCHER, M. J. The string-to-string correction problem. *Journal of the ACM*, 21(1):168–173, 1974.
- NEEDLEMAN, S. B.; WUNSCH, C. D. A general method applicable to the search for similarities in the amino acid sequence of two proteins. *Journal of Molecular Biology*, 48(3):443–453, 1970.
- HIRSCHBERG, D. S. A linear space algorithm for computing maximal common subsequences. *Communications of the ACM*, 18(6):341–343, 1975.